# Design a Reward Model

In [93]:
# imports

from pathlib import Path
import pickle
from collections import defaultdict, Counter
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from bisect import bisect_right
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
)
import copy
import pandas as pd

## Load predefined data

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

with open(DATA_DIR / "artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

print("Available artifacts:")
for key, value in artifacts.items():
    print(f"{key}: {type(value).__name__}")

Available artifacts:
user_sequences: dict
item_to_token: dict
token_to_item: dict
train_sequences: list
t_train: float64
t_validate: float64


In [6]:
with open(DATA_DIR / "reward_interactions.pkl", "rb") as f:
    reward_interactions = pickle.load(f)

for split_name, rows in reward_interactions.items():
    print(split_name, len(rows))
    if rows:
        print(rows[0])

train 194737
{'uid': 28700, 'timestamp': 46295, 'item_id': 7094195, 'played_ratio_pct': 47}
validation 41728
{'uid': 28700, 'timestamp': 20619030, 'item_id': 4976274, 'played_ratio_pct': 100}
test 41731
{'uid': 28700, 'timestamp': 23690465, 'item_id': 1918549, 'played_ratio_pct': 100}


In [5]:
with open(
    Path("./../data") / "artifacts.pkl", "rb"
) as f:
    artifacts = pickle.load(f)

user_sequences = artifacts["user_sequences"]
item_to_token = artifacts["item_to_token"]
token_to_item = artifacts["token_to_item"]
train_sequences = artifacts["train_sequences"]

t_train = artifacts["t_train"]
t_validate = artifacts["t_validate"]

## Prepare the dataset for the reward model

In [23]:
multi_event = load_dataset(
    "yandex/yambda",
    data_dir="flat/50m",
    data_files="multi_event.parquet",
    split="train",
)

print("Rows:", len(multi_event))
print("Columns:", multi_event.column_names)

# reuse the users
sampled_uids = set(user_sequences.keys())

selected_events = multi_event.filter(
    lambda row: row["uid"] in sampled_uids,
    desc="Filtering selected users",
)

print("Selected rows:", len(selected_events))
print("Event types:", Counter(selected_events["event_type"]))
print("Example:", selected_events[0])

flat/50m/multi_event.parquet: reconstructing file:   0%|          |  0.00B /  384MB            

flat/50m/multi_event.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Rows: 47790449
Columns: ['uid', 'timestamp', 'item_id', 'is_organic', 'played_ratio_pct', 'track_length_seconds', 'event_type']


Filtering selected users:   0%|          | 0/47790449 [00:00<?, ? examples/s]

Selected rows: 285554
Event types: Counter({'listen': 278196, 'like': 5433, 'unlike': 1199, 'dislike': 674, 'undislike': 52})
Example: {'uid': 28700, 'timestamp': 46295, 'item_id': 7094195, 'is_organic': 0, 'played_ratio_pct': 47, 'track_length_seconds': 255, 'event_type': 'listen'}


### Some diagnostics

In [36]:
# check if there is enough feedback in each split

split_event_counts = Counter()
split_vocab_counts = Counter()

for row in selected_events:
    timestamp = row["timestamp"]

    if timestamp < t_train:
        split = "train"
    elif timestamp < t_validate:
        split = "validation"
    else:
        split = "test"

    event_type = row["event_type"]
    split_event_counts[(split, event_type)] += 1

    if row["item_id"] in item_to_token:
        split_vocab_counts[(split, event_type)] += 1

for split in ("train", "validation", "test"):
    print(f"\n{split.upper()}")

    for event_type in ("listen", "like", "dislike", "unlike", "undislike"):
        total = split_event_counts[(split, event_type)]
        in_vocab = split_vocab_counts[(split, event_type)]

        print(
            f"{event_type:10s} total={total:6,} | "
            f"in vocabulary={in_vocab:6,}"
        )


TRAIN
listen     total=194,737 | in vocabulary=57,259
like       total= 3,037 | in vocabulary=   835
dislike    total=   230 | in vocabulary=    31
unlike     total=   923 | in vocabulary=   135
undislike  total=    37 | in vocabulary=     6

VALIDATION
listen     total=41,728 | in vocabulary= 6,728
like       total= 1,823 | in vocabulary=    80
dislike    total=   211 | in vocabulary=    11
unlike     total=   165 | in vocabulary=    20
undislike  total=     5 | in vocabulary=     1

TEST
listen     total=41,731 | in vocabulary= 5,424
like       total=   573 | in vocabulary=    72
dislike    total=   233 | in vocabulary=    10
unlike     total=   111 | in vocabulary=    10
undislike  total=    10 | in vocabulary=     1


In [37]:
# Count item frequency in training events.
train_item_counts = Counter(
    row["item_id"]
    for row in selected_events
    if row["timestamp"] < t_train
    and row["event_type"] in {"listen", "like", "dislike"}
)

print("Unique training items:", len(train_item_counts))

# Count how many training feedback events would be covered
# by the original vocabulary versus a larger vocabulary.
for event_type in ("like", "dislike"):
    rows = [
        row for row in selected_events
        if row["timestamp"] < t_train
        and row["event_type"] == event_type
    ]

    old_covered = sum(
        row["item_id"] in item_to_token for row in rows
    )

    print(
        f"{event_type}: total={len(rows):,}, "
        f"old vocabulary={old_covered:,}"
    )

Unique training items: 33133
like: total=3,037, old vocabulary=835
dislike: total=230, old vocabulary=31


Seems like we need to build a new vocabulary for the reward model!!! 1000 most common items we used in the baseline model are not enough :(

In [38]:
# build vocabulary from training events
train_item_counts = Counter(
    row["item_id"]
    for row in selected_events
    if row["timestamp"] < t_train
    and row["event_type"] in {"listen", "like", "dislike"}
)

REWARD_VOCAB_SIZE = min(10_000, len(train_item_counts))

reward_items = [
    item_id
    for item_id, count in train_item_counts.most_common(REWARD_VOCAB_SIZE)
]

reward_item_to_token = {
    item_id: token
    for token, item_id in enumerate(reward_items)
}

reward_token_to_item = {
    token: item_id
    for item_id, token in reward_item_to_token.items()
}

REWARD_PAD_ID = len(reward_item_to_token)

print("Reward vocabulary size:", len(reward_item_to_token))
print("Padding ID:", REWARD_PAD_ID)
print("Original vocabulary size:", len(item_to_token))

Reward vocabulary size: 10000
Padding ID: 10000
Original vocabulary size: 1000


In [40]:
# how much feedback in the new vocabulary?

for split in ("train", "validation", "test"):
    rows = [
        row for row in selected_events
        if (
            (split == "train" and row["timestamp"] < t_train)
            or (
                split == "validation"
                and t_train <= row["timestamp"] < t_validate
            )
            or (split == "test" and row["timestamp"] >= t_validate)
        )
    ]

    print(f"\n{split.upper()}")

    for event_type in ("listen", "like", "dislike"):
        feedback_rows = [
            row for row in rows
            if row["event_type"] == event_type
        ]

        covered = sum(
            row["item_id"] in reward_item_to_token
            for row in feedback_rows
        )

        print(
            f"{event_type:8s}: {covered:,}/{len(feedback_rows):,} "
            f"({covered / max(len(feedback_rows), 1):.1%})"
        )


TRAIN
listen  : 155,998/194,737 (80.1%)
like    : 2,811/3,037 (92.6%)
dislike : 139/230 (60.4%)

VALIDATION
listen  : 22,211/41,728 (53.2%)
like    : 316/1,823 (17.3%)
dislike : 47/211 (22.3%)

TEST
listen  : 18,064/41,731 (43.3%)
like    : 194/573 (33.9%)
dislike : 33/233 (14.2%)


How reward will be calculated:

1. User liked the track that he listened to for p%: 0.5 + 0.5p
2. User disliked the track that he listened to for p%: -1 + 0.5p
3. User listened to a track for p% without like/dislike: p

We use listen/like/dislike events for constructing examples (unlike/undislike are not considered). For like/dislike events we use the last percentage of the listen if available.

In [43]:
MAX_LENGTH = 32

examples = {
    "train": [],
    "validation": [],
    "test": [],
}

# recent listening history for each user.
histories = defaultdict(list)

# most recent listen of each track by each user
last_listen = {}

# process events in chronological order for each user
sorted_events = sorted(
    selected_events,
    key=lambda row: (row["uid"], row["timestamp"]),
)

for row in sorted_events:
    uid = row["uid"]
    item_id = row["item_id"]
    timestamp = row["timestamp"]
    event_type = row["event_type"]

    if timestamp < t_train:
        split = "train"
    elif timestamp < t_validate:
        split = "validation"
    else:
        split = "test"

    candidate = reward_item_to_token.get(item_id)

    # ignore candidate tracks outside the reward vocabulary
    if candidate is None:
        # still record listens for context only if the track has a token
        continue

    # create a listen example using only the preceding history
    if event_type == "listen":
        ratio = row["played_ratio_pct"]

        if ratio is not None and histories[uid]:
            p = float(np.clip(float(ratio) / 100.0, 0.0, 1.0))
            examples[split].append({
                "history": histories[uid][-MAX_LENGTH:].copy(),
                "candidate": candidate,
                "reward": p,
                "feedback": None,
                "source": "listen",
            })

        # save the current listen for potential later feedback matching
        last_listen[(uid, item_id)] = {
            "timestamp": timestamp,
            "played_ratio_pct": ratio,
        }

        # add this track after constructing its example
        histories[uid].append(candidate)
        histories[uid] = histories[uid][-MAX_LENGTH:]

    # create direct examples from explicit likes and dislikes
    elif event_type in {"like", "dislike"}:
        previous_listen = last_listen.get((uid, item_id))
        p = None

        if previous_listen is not None:
            previous_ratio = previous_listen["played_ratio_pct"]

            if (
                previous_listen["timestamp"] < timestamp
                and previous_ratio is not None
            ):
                p = float(np.clip(float(previous_ratio) / 100.0, 0.0, 1.0))

        if event_type == "like":
            reward = 0.5 + 0.5 * p if p is not None else 1.0
        else:
            reward = -1.0 + 0.5 * p if p is not None else -1.0

        examples[split].append({
            "history": histories[uid][-MAX_LENGTH:].copy(),
            "candidate": candidate,
            "reward": reward,
            "feedback": event_type,
            "source": "explicit_feedback",
        })

for split, rows in examples.items():
    print(f"\n{split.upper()}: {len(rows):,} examples")
    print("Sources:", {
        source: sum(row["source"] == source for row in rows)
        for source in ("listen", "explicit_feedback")
    })
    print("Feedback:", {
        feedback: sum(row["feedback"] == feedback for row in rows)
        for feedback in ("like", "dislike", None)
    })
    if rows:
        rewards = [row["reward"] for row in rows]
        print("Mean reward:", round(float(np.mean(rewards)), 4))
        print("Reward range:", (min(rewards), max(rewards)))


TRAIN: 158,903 examples
Sources: {'listen': 155953, 'explicit_feedback': 2950}
Feedback: {'like': 2811, 'dislike': 139, None: 155953}
Mean reward: 0.6479
Reward range: (-1.0, 1.0)

VALIDATION: 22,570 examples
Sources: {'listen': 22207, 'explicit_feedback': 363}
Feedback: {'like': 316, 'dislike': 47, None: 22207}
Mean reward: 0.6056
Reward range: (-1.0, 1.0)

TEST: 18,290 examples
Sources: {'listen': 18063, 'explicit_feedback': 227}
Feedback: {'like': 194, 'dislike': 33, None: 18063}
Mean reward: 0.5708
Reward range: (-1.0, 1.0)


In [44]:
for split, rows in examples.items():
    rewards = np.array([row["reward"] for row in rows])

    print(f"\n{split.upper()}")
    print("Count:", len(rewards))
    print("Mean:", round(rewards.mean(), 4))
    print("Std:", round(rewards.std(), 4))
    print("Negative rewards:", int((rewards < 0).sum()))
    print("Zero rewards:", int((rewards == 0).sum()))
    print("Positive rewards:", int((rewards > 0).sum()))


TRAIN
Count: 158903
Mean: 0.6479
Std: 0.4423
Negative rewards: 139
Zero rewards: 13143
Positive rewards: 145621

VALIDATION
Count: 22570
Mean: 0.6056
Std: 0.4578
Negative rewards: 47
Zero rewards: 2284
Positive rewards: 20239

TEST
Count: 18290
Mean: 0.5708
Std: 0.4634
Negative rewards: 33
Zero rewards: 2438
Positive rewards: 15819


In [47]:
class RewardDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

def collate_reward_batch(batch):
    max_len = max(len(row["history"]) for row in batch)

    histories = torch.full(
        (len(batch), max_len), REWARD_PAD_ID, dtype=torch.long
    )
    masks = torch.zeros((len(batch), max_len), dtype=torch.float32)

    for i, row in enumerate(batch):
        history = torch.tensor(row["history"], dtype=torch.long)
        histories[i, :len(history)] = history
        masks[i, :len(history)] = 1.0

    return {
        "history": histories,
        "history_mask": masks,
        "candidate": torch.tensor(
            [row["candidate"] for row in batch], dtype=torch.long
        ),
        "reward": torch.tensor(
            [row["reward"] for row in batch], dtype=torch.float32
        ),
    }
    

In [57]:
# balance the like/dislike examples

train_dataset = RewardDataset(examples["train"])
validation_dataset = RewardDataset(examples["validation"])
test_dataset = RewardDataset(examples["test"])

# desired sampling proportions for each example type
target_proportions = {
    "listen": 0.90,
    "like": 0.09,
    "dislike": 0.01,
}

# count the examples in each group
group_counts = {
    "listen": sum(row["feedback"] is None for row in examples["train"]),
    "like": sum(row["feedback"] == "like" for row in examples["train"]),
    "dislike": sum(row["feedback"] == "dislike" for row in examples["train"]),
}

# give each example a weight based on its group
sample_weights = []
for row in examples["train"]:
    if row["feedback"] is None:
        group = "listen"
    else:
        group = row["feedback"]

    weight = target_proportions[group] / group_counts[group]
    sample_weights.append(weight)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    sampler=sampler,
    collate_fn=collate_reward_batch,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_reward_batch,
)

# Test: natural distribution, no sampling
test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_reward_batch,
)

print("Training examples:", len(train_dataset))
print("Group counts:", group_counts)
print("Sampler target proportions:", target_proportions)

Training examples: 158903
Group counts: {'listen': 155953, 'like': 2811, 'dislike': 139}
Sampler target proportions: {'listen': 0.9, 'like': 0.09, 'dislike': 0.01}


In [53]:
with open(DATA_DIR / "reward_examples.pkl", "wb") as f:
    pickle.dump(examples, f)

## [FIRST EXPERIMENT. REWARD MODEL 1]

In [58]:
# the model

class RewardModel(nn.Module):
    def __init__(self, n_items, embedding_dim=64, hidden_dim=128):
        super().__init__()
        self.item_embedding = nn.Embedding(
            n_items + 1, embedding_dim, padding_idx=REWARD_PAD_ID
        )
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Tanh(),
        )

    def forward(self, history, history_mask, candidate):
        embeddings = self.item_embedding(history)
        mask = history_mask.unsqueeze(-1)

        history_vector = (embeddings * mask).sum(dim=1)
        history_length = mask.sum(dim=1).clamp(min=1.0)
        history_vector = history_vector / history_length

        candidate_vector = self.item_embedding(candidate)
        combined = torch.cat([history_vector, candidate_vector], dim=1)
        return self.mlp(combined).squeeze(-1)

In [59]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

In [51]:
def run_epoch(loader, training=False):
    reward_model.train(training)
    total_loss = 0.0
    total_count = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.set_grad_enabled(training):
            predictions = reward_model(
                batch["history"],
                batch["history_mask"],
                batch["candidate"],
            )
            loss = loss_fn(predictions, batch["reward"])

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
                optimizer.step()

        n = len(batch["reward"])
        total_loss += loss.item() * n
        total_count += n

    return total_loss / max(total_count, 1)

In [ ]:
# initialize the reward model
reward_model = RewardModel(
    n_items=len(reward_item_to_token),
    embedding_dim=64,
    hidden_dim=128,
).to(device)

optimizer = torch.optim.AdamW(
    reward_model.parameters(),
    lr=3e-4,
)

loss_fn = nn.MSELoss()

# training settings
EPOCHS = 15
PATIENCE = 3

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(validation_loader, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"Train MSE: {train_loss:.4f} | "
        f"Validation MSE: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(reward_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    # try to avoid overfitting
    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

# restore the checkpoint with the best validation loss
if best_state is not None:
    reward_model.load_state_dict(best_state)

print(f"Best validation MSE: {best_val_loss:.4f}")

Epoch 01 | Train MSE: 0.2421 | Validation MSE: 0.1979
Epoch 02 | Train MSE: 0.1677 | Validation MSE: 0.1905
Epoch 03 | Train MSE: 0.1540 | Validation MSE: 0.1891
Epoch 04 | Train MSE: 0.1452 | Validation MSE: 0.1896
Epoch 05 | Train MSE: 0.1372 | Validation MSE: 0.1927
Epoch 06 | Train MSE: 0.1316 | Validation MSE: 0.1944
Early stopping.
Best validation MSE: 0.1891


## Test the Reward Model

In [61]:
reward_model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        predictions = reward_model(
            batch["history"],
            batch["history_mask"],
            batch["candidate"],
        )

        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(batch["reward"].cpu().numpy())

predictions = np.asarray(all_predictions)
targets = np.asarray(all_targets)

# the test loader uses shuffle=False, so this order matches examples["test"].
feedback_groups = np.array([
    row["feedback"] if row["feedback"] is not None else "listen"
    for row in examples["test"]
])

def print_metrics(name, mask, preds, actual):
    if mask.sum() == 0:
        print(f"{name}: no examples")
        return

    errors = preds[mask] - actual[mask]
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    print(
        f"{name:10s} | n={mask.sum():5d} | "
        f"MAE={mae:.4f} | RMSE={rmse:.4f}"
    )

print("Reward model performance")
print_metrics("Overall", np.ones(len(targets), dtype=bool),
              predictions, targets)

for group in ["listen", "like", "dislike"]:
    print_metrics(
        group,
        feedback_groups == group,
        predictions,
        targets,
    )

Reward model performance
Overall    | n=18290 | MAE=0.3906 | RMSE=0.4459
listen     | n=18063 | MAE=0.3891 | RMSE=0.4421
like       | n=  194 | MAE=0.3437 | RMSE=0.4049
dislike    | n=   33 | MAE=1.4842 | RMSE=1.4996


In [ ]:
# baseline: always predict the mean reward in the training data
train_mean_reward = np.mean([
    row["reward"] for row in examples["train"]
])
baseline_predictions = np.full_like(targets, train_mean_reward)

print("\nConstant-mean baseline")
print_metrics(
    "Overall",
    np.ones(len(targets), dtype=bool),
    baseline_predictions,
    targets,
)


Constant-mean baseline
Overall    | n=18290 | MAE=0.4399 | RMSE=0.4698


In [63]:
for group in ["listen", "like", "dislike"]:
    mask = feedback_groups == group

    print(f"\n{group.upper()} (n={mask.sum()})")
    print(f"  Actual mean:      {targets[mask].mean():.4f}")
    print(f"  Predicted mean:   {predictions[mask].mean():.4f}")
    print(f"  Actual min/max:   {targets[mask].min():.4f} / {targets[mask].max():.4f}")
    print(f"  Predicted min/max:{predictions[mask].min():.4f} / {predictions[mask].max():.4f}")
    print(f"  Predicted negative: {(predictions[mask] < 0).sum()}/{mask.sum()}")


LISTEN (n=18063)
  Actual mean:      0.5706
  Predicted mean:   0.6114
  Actual min/max:   0.0000 / 1.0000
  Predicted min/max:-0.4344 / 0.9835
  Predicted negative: 151/18063

LIKE (n=194)
  Actual mean:      0.8514
  Predicted mean:   0.5697
  Actual min/max:   0.5000 / 1.0000
  Predicted min/max:-0.2309 / 0.9042
  Predicted negative: 1/194

DISLIKE (n=33)
  Actual mean:      -0.9579
  Predicted mean:   0.5263
  Actual min/max:   -1.0000 / -0.5050
  Predicted min/max:-0.0865 / 0.9060
  Predicted negative: 2/33


The model fails to predict dislikes because it tries to minimize loss while having too little training examples for the dislike events.

## [SECOND EXPERIMENT. REWARD MODEL 2]

We will create a separate feedback signal for the dislikes specifically.

In [75]:
def collate_reward_batch2(batch):
    max_len = max(len(row["history"]) for row in batch)

    histories = torch.full(
        (len(batch), max_len),
        REWARD_PAD_ID,
        dtype=torch.long,
    )
    masks = torch.zeros(
        (len(batch), max_len),
        dtype=torch.float32,
    )

    for i, row in enumerate(batch):
        history = torch.tensor(row["history"], dtype=torch.long)
        histories[i, :len(history)] = history
        masks[i, :len(history)] = 1.0

    return {
        "history": histories,
        "history_mask": masks,
        "candidate": torch.tensor(
            [row["candidate"] for row in batch],
            dtype=torch.long,
        ),
        "reward": torch.tensor(
            [row["reward"] for row in batch],
            dtype=torch.float32,
        ),
        "is_dislike": torch.tensor(
            [row["feedback"] == "dislike" for row in batch],
            dtype=torch.float32,
        ),
    }

In [64]:
class MultiTaskRewardModel(nn.Module):
    def __init__(self, n_items, embedding_dim=64, hidden_dim=128):
        super().__init__()

        self.item_embedding = nn.Embedding(
            n_items + 1,
            embedding_dim,
            padding_idx=REWARD_PAD_ID,
        )

        self.shared = nn.Sequential(
            nn.Linear(embedding_dim * 2, hidden_dim),
            nn.ReLU(),
        )

        # numerical reward prediction in [-1, 1]
        self.reward_head = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh(),
        )

        # raw logit for dislike classification
        self.dislike_head = nn.Linear(hidden_dim, 1)

    def forward(self, history, history_mask, candidate):
        embeddings = self.item_embedding(history)
        mask = history_mask.unsqueeze(-1)

        history_vector = (embeddings * mask).sum(dim=1)
        history_length = mask.sum(dim=1).clamp(min=1.0)
        history_vector = history_vector / history_length

        candidate_vector = self.item_embedding(candidate)
        combined = torch.cat(
            [history_vector, candidate_vector],
            dim=1,
        )

        shared_features = self.shared(combined)

        reward_prediction = self.reward_head(
            shared_features
        ).squeeze(-1)

        dislike_logit = self.dislike_head(
            shared_features
        ).squeeze(-1)

        return reward_prediction, dislike_logit

In [76]:
# dataloaders

train_loader2 = DataLoader(
    train_dataset,
    batch_size=256,
    sampler=sampler,
    collate_fn=collate_reward_batch2,
)

validation_loader2 = DataLoader(
    validation_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_reward_batch2,
)

test_loader2 = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_reward_batch2,
)

In [77]:
reward_model2 = MultiTaskRewardModel(
    n_items=len(reward_item_to_token),
    embedding_dim=64,
    hidden_dim=128,
).to(device)

optimizer2 = torch.optim.AdamW(
    reward_model2.parameters(),
    lr=3e-4,
)

reward_loss_fn = nn.MSELoss()

# give positive-class (dislike) errors more weight
n_dislikes = group_counts["dislike"]
n_non_dislikes = group_counts["listen"] + group_counts["like"]

pos_weight = torch.tensor(
    [n_non_dislikes / n_dislikes],
    dtype=torch.float32,
    device=device,
)

dislike_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight,
)

print("Dislike positive-class weight:", pos_weight.item())

Dislike positive-class weight: 1142.18701171875


In [78]:
# Replace the heavily weighted classification loss.
dislike_loss_fn = nn.BCEWithLogitsLoss()

# Reduce the classification objective's influence initially.
LAMBDA_DISLIKE = 0.25

# Reinitialize model and optimizer to avoid continuing from the
# unstable model's learned weights.
reward_model2 = MultiTaskRewardModel(
    n_items=len(reward_item_to_token),
    embedding_dim=64,
    hidden_dim=128,
).to(device)

optimizer2 = torch.optim.AdamW(
    reward_model2.parameters(),
    lr=1e-4,
)

print("Classification loss: unweighted BCE")
print("Classification loss coefficient:", LAMBDA_DISLIKE)

Classification loss: unweighted BCE
Classification loss coefficient: 0.25


Now the loss function is L = L_reward + lambda * L_dislike

In [79]:
LAMBDA_DISLIKE = 0.25
EPOCHS = 15
PATIENCE = 3

def run_epoch2(loader, training=False):
    reward_model2.train(training)

    total_loss = 0.0
    total_count = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.set_grad_enabled(training):
            reward_pred, dislike_logit = reward_model2(
                batch["history"],
                batch["history_mask"],
                batch["candidate"],
            )

            reward_loss = reward_loss_fn(
                reward_pred,
                batch["reward"],
            )

            dislike_loss = dislike_loss_fn(
                dislike_logit,
                batch["is_dislike"],
            )

            loss = reward_loss + LAMBDA_DISLIKE * dislike_loss

            if training:
                optimizer2.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    reward_model2.parameters(),
                    1.0,
                )
                optimizer2.step()

        n = len(batch["reward"])
        total_loss += loss.item() * n
        total_count += n

    return total_loss / max(total_count, 1)


best_val_loss2 = float("inf")
best_state2 = None
patience_counter2 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch2(train_loader2, training=True)
    val_loss = run_epoch2(validation_loader2, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss2:
        best_val_loss2 = val_loss
        best_state2 = copy.deepcopy(reward_model2.state_dict())
        patience_counter2 = 0
    else:
        patience_counter2 += 1

    if patience_counter2 >= PATIENCE:
        print("Early stopping.")
        break

if best_state2 is not None:
    reward_model2.load_state_dict(best_state2)

print(f"Best validation loss: {best_val_loss2:.4f}")

Epoch 01 | Train loss: 0.3462 | Validation loss: 0.2569
Epoch 02 | Train loss: 0.2233 | Validation loss: 0.2211
Epoch 03 | Train loss: 0.1980 | Validation loss: 0.2099
Epoch 04 | Train loss: 0.1867 | Validation loss: 0.2053
Epoch 05 | Train loss: 0.1788 | Validation loss: 0.2033
Epoch 06 | Train loss: 0.1740 | Validation loss: 0.2010
Epoch 07 | Train loss: 0.1673 | Validation loss: 0.1996
Epoch 08 | Train loss: 0.1627 | Validation loss: 0.1993
Epoch 09 | Train loss: 0.1576 | Validation loss: 0.1987
Epoch 10 | Train loss: 0.1559 | Validation loss: 0.1986
Epoch 11 | Train loss: 0.1512 | Validation loss: 0.1990
Epoch 12 | Train loss: 0.1478 | Validation loss: 0.1982
Epoch 13 | Train loss: 0.1446 | Validation loss: 0.1984
Epoch 14 | Train loss: 0.1415 | Validation loss: 0.1996
Epoch 15 | Train loss: 0.1401 | Validation loss: 0.1994
Early stopping.
Best validation loss: 0.1982


In [81]:

reward_model2.eval()

reward_preds = []
reward_targets = []
dislike_probs = []
dislike_targets = []

with torch.no_grad():
    for batch in test_loader2:
        batch = {k: v.to(device) for k, v in batch.items()}

        reward_pred, dislike_logit = reward_model2(
            batch["history"],
            batch["history_mask"],
            batch["candidate"],
        )

        reward_preds.extend(reward_pred.cpu().numpy())
        reward_targets.extend(batch["reward"].cpu().numpy())

        probabilities = torch.sigmoid(dislike_logit)
        dislike_probs.extend(probabilities.cpu().numpy())
        dislike_targets.extend(batch["is_dislike"].cpu().numpy())

reward_preds = np.asarray(reward_preds)
reward_targets = np.asarray(reward_targets)
dislike_probs = np.asarray(dislike_probs)
dislike_targets = np.asarray(dislike_targets).astype(int)

# 1. Reward regression metrics
mae = np.mean(np.abs(reward_preds - reward_targets))
rmse = np.sqrt(np.mean((reward_preds - reward_targets) ** 2))

print("REWARD REGRESSION")
print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# 2. Dislike classification metrics at threshold 0.5
threshold = 0.5
dislike_preds = (dislike_probs >= threshold).astype(int)

print("\nDISLIKE CLASSIFICATION")
print(f"Actual dislikes:       {dislike_targets.sum()}")
print(f"Predicted dislikes:    {dislike_preds.sum()}")
print(f"Precision: {precision_score(dislike_targets, dislike_preds, zero_division=0):.4f}")
print(f"Recall:    {recall_score(dislike_targets, dislike_preds, zero_division=0):.4f}")
print(f"F1:        {f1_score(dislike_targets, dislike_preds, zero_division=0):.4f}")

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(dislike_targets, dislike_preds, labels=[0, 1]))

if dislike_targets.sum() > 0:
    ap = average_precision_score(dislike_targets, dislike_probs)
    print(f"\nAverage precision (PR-AUC summary): {ap:.4f}")

print("\nPredicted dislike probability")
print(f"Mean for dislikes:     {dislike_probs[dislike_targets == 1].mean():.4f}")
print(f"Mean for non-dislikes: {dislike_probs[dislike_targets == 0].mean():.4f}")

REWARD REGRESSION
Test MAE:  0.3967
Test RMSE: 0.4567

DISLIKE CLASSIFICATION
Actual dislikes:       33
Predicted dislikes:    0
Precision: 0.0000
Recall:    0.0000
F1:        0.0000

Confusion matrix [[TN, FP], [FN, TP]]:
[[18257     0]
 [   33     0]]

Average precision (PR-AUC summary): 0.0024

Predicted dislike probability
Mean for dislikes:     0.0101
Mean for non-dislikes: 0.0089


In [82]:
# explore the classication set performance to choose a better thereshold

reward_model2.eval()

val_probs = []
val_labels = []

with torch.no_grad():
    for batch in validation_loader2:
        batch = {k: v.to(device) for k, v in batch.items()}

        _, dislike_logit = reward_model2(
            batch["history"],
            batch["history_mask"],
            batch["candidate"],
        )

        val_probs.extend(
            torch.sigmoid(dislike_logit).cpu().numpy()
        )
        val_labels.extend(
            batch["is_dislike"].cpu().numpy()
        )

val_probs = np.asarray(val_probs)
val_labels = np.asarray(val_labels).astype(int)

print("Validation dislikes:", val_labels.sum())
print("Validation prevalence:", round(val_labels.mean(), 5))
print("Average precision:", round(
    average_precision_score(val_labels, val_probs), 5
))

for group, mask in [
    ("Dislikes", val_labels == 1),
    ("Non-dislikes", val_labels == 0),
]:
    values = val_probs[mask]
    print(f"\n{group} (n={len(values)})")
    print("Mean probability:", round(values.mean(), 5))
    print("Median probability:", round(np.median(values), 5))
    print("90th percentile:", round(np.quantile(values, 0.90), 5))
    print("Maximum probability:", round(values.max(), 5))

Validation dislikes: 47
Validation prevalence: 0.00208
Average precision: 0.00581

Dislikes (n=47)
Mean probability: 0.0135
Median probability: 0.00759
90th percentile: 0.0301
Maximum probability: 0.07701

Non-dislikes (n=22523)
Mean probability: 0.00698
Median probability: 0.00403
90th percentile: 0.0163
Maximum probability: 0.15365


In [83]:
print("Threshold | Precision | Recall | F1")

for threshold in [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.10, 0.50]:
    preds = (val_probs >= threshold).astype(int)

    precision = precision_score(
        val_labels, preds, zero_division=0
    )
    recall = recall_score(
        val_labels, preds, zero_division=0
    )
    f1 = f1_score(
        val_labels, preds, zero_division=0
    )

    print(
        f"{threshold:9.3f} | {precision:9.4f} | "
        f"{recall:6.4f} | {f1:.4f}"
    )

Threshold | Precision | Recall | F1
    0.001 |    0.0023 | 0.9574 | 0.0046
    0.002 |    0.0026 | 0.9149 | 0.0053
    0.005 |    0.0032 | 0.6596 | 0.0064
    0.010 |    0.0044 | 0.4468 | 0.0087
    0.020 |    0.0059 | 0.1915 | 0.0114
    0.050 |    0.0145 | 0.0426 | 0.0216
    0.100 |    0.0000 | 0.0000 | 0.0000
    0.500 |    0.0000 | 0.0000 | 0.0000


The model will still fail to classify most dislikes even if we lower the threshold.

## [THIRD EXPERIMENT. SEPARATE SAMPLER FOR CLASSIFICATION]

In [84]:
train_rows2 = examples["train"]

n_dislikes = sum(row["feedback"] == "dislike" for row in train_rows2)
n_non_dislikes = len(train_rows2) - n_dislikes

classification_proportions = {
    "non_dislike": 0.70,
    "dislike": 0.30,
}

weights2 = []
for row in train_rows2:
    if row["feedback"] == "dislike":
        group = "dislike"
    else:
        group = "non_dislike"

    weight = (
        classification_proportions[group]
        / (n_dislikes if group == "dislike" else n_non_dislikes)
    )
    weights2.append(weight)

sampler2 = WeightedRandomSampler(
    weights=torch.tensor(weights2, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
)

train_loader3 = DataLoader(
    train_dataset,
    batch_size=256,
    sampler=sampler2,
    collate_fn=collate_reward_batch2,
)

print("Dislikes in training data:", n_dislikes)
print("Non-dislikes in training data:", n_non_dislikes)
print("Target sampling proportions:", classification_proportions)

Dislikes in training data: 139
Non-dislikes in training data: 158764
Target sampling proportions: {'non_dislike': 0.7, 'dislike': 0.3}


In [87]:
LAMBDA_DISLIKE = 1.0

reward_model3 = MultiTaskRewardModel(
    n_items=len(reward_item_to_token),
    embedding_dim=64,
    hidden_dim=128,
).to(device)

optimizer3 = torch.optim.AdamW(
    reward_model3.parameters(),
    lr=1e-4,
)

reward_loss_fn = nn.MSELoss()
dislike_loss_fn = nn.BCEWithLogitsLoss()

In [88]:
# train the third model

EPOCHS = 15
PATIENCE = 3

LAMBDA_DISLIKE = 1.0

def run_epoch3(loader, training=False):
    reward_model3.train(training)

    total_loss = 0.0
    total_count = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.set_grad_enabled(training):
            reward_pred, dislike_logit = reward_model3(
                batch["history"],
                batch["history_mask"],
                batch["candidate"],
            )

            reward_loss = reward_loss_fn(
                reward_pred,
                batch["reward"],
            )

            dislike_loss = dislike_loss_fn(
                dislike_logit,
                batch["is_dislike"],
            )

            loss = reward_loss + LAMBDA_DISLIKE * dislike_loss

            if training:
                optimizer3.zero_grad()
                loss.backward()

                nn.utils.clip_grad_norm_(
                    reward_model3.parameters(),
                    max_norm=1.0,
                )

                optimizer3.step()

        n = len(batch["reward"])
        total_loss += loss.item() * n
        total_count += n

    return total_loss / max(total_count, 1)


# early stopping setup
best_val_loss3 = float("inf")
best_state3 = None
patience_counter3 = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch3(train_loader3, training=True)
    val_loss = run_epoch3(validation_loader2, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss: {train_loss:.4f} | "
        f"Validation loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss3:
        best_val_loss3 = val_loss
        best_state3 = copy.deepcopy(reward_model3.state_dict())
        patience_counter3 = 0
    else:
        patience_counter3 += 1

    if patience_counter3 >= PATIENCE:
        print("Early stopping.")
        break


# restore the best checkpoint
if best_state3 is not None:
    reward_model3.load_state_dict(best_state3)

print(f"Best validation loss: {best_val_loss3:.4f}")

Epoch 01 | Train loss: 1.0370 | Validation loss: 0.6204
Epoch 02 | Train loss: 0.5313 | Validation loss: 0.4382
Epoch 03 | Train loss: 0.3060 | Validation loss: 0.3438
Epoch 04 | Train loss: 0.2214 | Validation loss: 0.2995
Epoch 05 | Train loss: 0.1833 | Validation loss: 0.2766
Epoch 06 | Train loss: 0.1637 | Validation loss: 0.2675
Epoch 07 | Train loss: 0.1520 | Validation loss: 0.2582
Epoch 08 | Train loss: 0.1423 | Validation loss: 0.2539
Epoch 09 | Train loss: 0.1354 | Validation loss: 0.2490
Epoch 10 | Train loss: 0.1312 | Validation loss: 0.2465
Epoch 11 | Train loss: 0.1268 | Validation loss: 0.2449
Epoch 12 | Train loss: 0.1226 | Validation loss: 0.2427
Epoch 13 | Train loss: 0.1193 | Validation loss: 0.2422
Epoch 14 | Train loss: 0.1162 | Validation loss: 0.2429
Epoch 15 | Train loss: 0.1151 | Validation loss: 0.2402
Best validation loss: 0.2402


In [ ]:
# evaluate the third model

reward_model3.eval()

reward_preds = []
reward_targets = []
dislike_probs = []
dislike_targets = []

with torch.no_grad():
    for batch in test_loader2:
        batch = {k: v.to(device) for k, v in batch.items()}

        reward_pred, dislike_logit = reward_model3(
            batch["history"],
            batch["history_mask"],
            batch["candidate"],
        )

        reward_preds.extend(reward_pred.cpu().numpy())
        reward_targets.extend(batch["reward"].cpu().numpy())

        probabilities = torch.sigmoid(dislike_logit)
        dislike_probs.extend(probabilities.cpu().numpy())
        dislike_targets.extend(batch["is_dislike"].cpu().numpy())

reward_preds = np.asarray(reward_preds)
reward_targets = np.asarray(reward_targets)
dislike_probs = np.asarray(dislike_probs)
dislike_targets = np.asarray(dislike_targets).astype(int)


# 1. reward regression metrics
mae = np.mean(np.abs(reward_preds - reward_targets))
rmse = np.sqrt(np.mean((reward_preds - reward_targets) ** 2))

print("REWARD REGRESSION")
print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")


# 2. dislike classification metrics
threshold = 0.5
dislike_preds = (dislike_probs >= threshold).astype(int)

print("\nDISLIKE CLASSIFICATION")
print(f"Actual dislikes:    {dislike_targets.sum()}")
print(f"Predicted dislikes: {dislike_preds.sum()}")
print(
    f"Precision: {precision_score(dislike_targets, dislike_preds, zero_division=0):.4f}"
)
print(
    f"Recall:    {recall_score(dislike_targets, dislike_preds, zero_division=0):.4f}"
)
print(
    f"F1:        {f1_score(dislike_targets, dislike_preds, zero_division=0):.4f}"
)

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(dislike_targets, dislike_preds, labels=[0, 1]))

if dislike_targets.sum() > 0:
    ap = average_precision_score(dislike_targets, dislike_probs)
    print(f"\nAverage precision (AP): {ap:.4f}")

print("\nPredicted dislike probability")
if np.any(dislike_targets == 1):
    print(
        f"Mean for dislikes:     "
        f"{dislike_probs[dislike_targets == 1].mean():.4f}"
    )

if np.any(dislike_targets == 0):
    print(
        f"Mean for non-dislikes: "
        f"{dislike_probs[dislike_targets == 0].mean():.4f}"
    )

REWARD REGRESSION
Test MAE:  0.4016
Test RMSE: 0.4676

DISLIKE CLASSIFICATION
Actual dislikes:    33
Predicted dislikes: 17
Precision: 0.0000
Recall:    0.0000
F1:        0.0000

Confusion matrix [[TN, FP], [FN, TP]]:
[[18240    17]
 [   33     0]]

Average precision (AP): 0.0023

Predicted dislike probability
Mean for dislikes:     0.0006
Mean for non-dislikes: 0.0023


## Compare Reward Models

In [92]:
# create a new version of the 1st model that is trained on extended vocabulary like the other two

# original regression architecture, using the 10k-item vocabulary

reward_model1_eval = RewardModel(
    n_items=len(reward_item_to_token),
    embedding_dim=64,
    hidden_dim=128,
).to(device)

optimizer1_eval = torch.optim.AdamW(
    reward_model1_eval.parameters(),
    lr=3e-4,
)

reward_loss_fn = nn.MSELoss()

EPOCHS = 10
PATIENCE = 3


def run_epoch1_eval(loader, training=False):
    reward_model1_eval.train(training)

    total_loss = 0.0
    total_count = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.set_grad_enabled(training):
            reward_pred = reward_model1_eval(
                batch["history"],
                batch["history_mask"],
                batch["candidate"],
            )

            loss = reward_loss_fn(reward_pred, batch["reward"])

            if training:
                optimizer1_eval.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    reward_model1_eval.parameters(), 1.0
                )
                optimizer1_eval.step()

        n = len(batch["reward"])
        total_loss += loss.item() * n
        total_count += n

    return total_loss / max(total_count, 1)


best_val_loss1_eval = float("inf")
best_state1_eval = None
patience_counter1_eval = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch1_eval(train_loader2, training=True)
    val_loss = run_epoch1_eval(validation_loader2, training=False)

    print(
        f"Epoch {epoch:02d} | "
        f"Train MSE: {train_loss:.4f} | "
        f"Validation MSE: {val_loss:.4f}"
    )

    if val_loss < best_val_loss1_eval:
        best_val_loss1_eval = val_loss
        best_state1_eval = copy.deepcopy(
            reward_model1_eval.state_dict()
        )
        patience_counter1_eval = 0
    else:
        patience_counter1_eval += 1

    if patience_counter1_eval >= PATIENCE:
        print("Early stopping.")
        break

reward_model1_eval.load_state_dict(best_state1_eval)
print(f"Best validation MSE: {best_val_loss1_eval:.4f}")

Epoch 01 | Train MSE: 0.2094 | Validation MSE: 0.1975
Epoch 02 | Train MSE: 0.1680 | Validation MSE: 0.1913
Epoch 03 | Train MSE: 0.1540 | Validation MSE: 0.1916
Epoch 04 | Train MSE: 0.1437 | Validation MSE: 0.1904
Epoch 05 | Train MSE: 0.1369 | Validation MSE: 0.1915
Epoch 06 | Train MSE: 0.1300 | Validation MSE: 0.1930
Epoch 07 | Train MSE: 0.1249 | Validation MSE: 0.1952
Early stopping.
Best validation MSE: 0.1904


In [94]:
# evaluate all three models and the mean baseline

def evaluate_reward_model(model, loader, multitask=False):
    model.eval()
    predictions = []
    targets = []
    dislike_probs = []
    dislike_targets = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            output = model(
                batch["history"],
                batch["history_mask"],
                batch["candidate"],
            )

            if multitask:
                reward_pred, dislike_logit = output
                dislike_probs.extend(
                    torch.sigmoid(dislike_logit).cpu().numpy()
                )
                dislike_targets.extend(
                    batch["is_dislike"].cpu().numpy()
                )
            else:
                reward_pred = output

            predictions.extend(reward_pred.cpu().numpy())
            targets.extend(batch["reward"].cpu().numpy())

    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    results = {
        "MAE": np.mean(np.abs(predictions - targets)),
        "RMSE": np.sqrt(np.mean((predictions - targets) ** 2)),
    }

    if multitask:
        dislike_targets = np.asarray(dislike_targets).astype(int)
        dislike_probs = np.asarray(dislike_probs)

        results["Dislike AP"] = (
            average_precision_score(dislike_targets, dislike_probs)
            if dislike_targets.sum() > 0
            else np.nan
        )

    return results


# evaluate each model on the same test examples
results = {
    "Model 1 — original architecture": evaluate_reward_model(
        reward_model1_eval, test_loader2
    ),
    "Model 2 — multitask": evaluate_reward_model(
        reward_model2, test_loader2, multitask=True
    ),
    "Model 3 — oversampled dislikes": evaluate_reward_model(
        reward_model3, test_loader2, multitask=True
    ),
}

# constant-mean baseline: use training rewards only
train_mean_reward = np.mean([
    row["reward"] for row in examples["train"]
])

test_targets = np.concatenate([
    batch["reward"].numpy()
    for batch in test_loader2
])

mean_predictions = np.full_like(test_targets, train_mean_reward)

results["Mean baseline"] = {
    "MAE": np.mean(np.abs(mean_predictions - test_targets)),
    "RMSE": np.sqrt(np.mean((mean_predictions - test_targets) ** 2)),
    "Dislike AP": np.nan,
}

comparison = pd.DataFrame(results).T
print(f"Training mean reward: {train_mean_reward:.4f}")
display(comparison.round(4))

Training mean reward: 0.6479


,MAE,RMSE,Dislike AP
Model 1 — original architecture,0.3865,0.4493,NaN
Model 2 — multitask,0.4160,0.5100,0.0020
Model 3 — oversampled dislikes,0.4016,0.4676,0.0023
Mean baseline,0.4399,0.4698,NaN


In [95]:
REWARD_MODEL_PATH = DATA_DIR / "reward_model1_best.pt"

torch.save(
    {
        "model_state_dict": reward_model1_eval.state_dict(),
        "n_items": len(reward_item_to_token),
        "embedding_dim": 64,
        "hidden_dim": 128,
        "reward_pad_id": REWARD_PAD_ID,
        "item_to_token": reward_item_to_token,
        "token_to_item": reward_token_to_item,
        "best_validation_mse": best_val_loss1_eval,
        "test_mae": 0.3865,
        "test_rmse": 0.4493,
    },
    REWARD_MODEL_PATH,
)

print(f"Saved reward model to: {REWARD_MODEL_PATH}")

Saved reward model to: /Users/fountainer/tlab/awesomeRecs/data/reward_model1_best.pt


In [96]:
checkpoint = torch.load(
    REWARD_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

reward_model_check = RewardModel(
    n_items=checkpoint["n_items"],
    embedding_dim=checkpoint["embedding_dim"],
    hidden_dim=checkpoint["hidden_dim"],
).to(device)

reward_model_check.load_state_dict(checkpoint["model_state_dict"])
reward_model_check.eval()

print("Reward model loaded successfully.")

Reward model loaded successfully.
